# Notebook 2 — Ablation Studies (Aligned with Final Training)
**RSNA Intracranial Hemorrhage Detection — Improved Pipeline**

Ablation study = controlled experiments where we remove **one** component at a time
from the final training recipe and measure AUC impact.

Final recipe (reference):
- Backbone: `tf_efficientnet_b4`
- Input: `380×380`, 9-channel 2.5D cache
- Classes: 6 (`any` + 5 subtypes)
- Loss: weighted BCE-with-logits style (matching NB01)
- Hard negative mining: ON (top-500, boost 3×)

## Experiments (single-variable changes only)
| # | Experiment | Control | Treatment |
|---|------------|---------|----------|
| 1 | **2.5D context** | 3ch center-only (no 2.5D) | 9ch 2.5D (baseline) |
| 2 | **Input resolution** | 256×256 | 380×380 (baseline) |
| 3 | **Hard negative mining** | mining OFF | mining ON (baseline) |

The baseline configuration is reused across comparisons to keep experiments fair.




In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  CONFIG (aligned to 01_training final recipe)  ██
# ══════════════════════════════════════════════════════════════════════════
from pathlib import Path
import os, json, gc, time, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ─── Paths (match your current improved pipeline outputs) ─────────────
NB02_DIR = Path('E:/major_8/major_2nd_cache_files')
MANIFEST_PATH  = NB02_DIR / 'manifest.csv'
NPY_CACHE_DIR  = NB02_DIR / 'cache'
NORM_STATS     = NB02_DIR / 'normalization_stats.json'

OUTPUT_DIR = Path('E:/major_8/majr_1_out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Ablation runtime controls (kept small for practical runs) ───────
ABLATION_FRAC = 0.15   # train subset fraction
ABLATION_EP   = 3      # epochs per experiment
SEED          = 42
FOLD_ID       = 0
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

# ─── Final model recipe from 01_training ─────────────────────────────
BACKBONE      = 'tf_efficientnet_b4'
BASE_IMG_SIZE = 380
IN_CHANNELS   = 9
N_CLASSES     = 6
BATCH_SIZE    = 8
LR            = 1.5e-4
ANY_WEIGHT    = 1.0
SUB_WEIGHT    = 0.4
LABEL_SMOOTH  = 0.05
MINE_TOP_K    = 500
MINE_BOOST    = 3.0

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']


def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

assert MANIFEST_PATH.exists(), f'manifest.csv missing: {MANIFEST_PATH}'
assert NPY_CACHE_DIR.exists(), f'cache dir missing: {NPY_CACHE_DIR}'

print(f'Device: {DEVICE}')
print(f'Backbone: {BACKBONE}, input={BASE_IMG_SIZE}, channels={IN_CHANNELS}, classes={N_CLASSES}')
print(f'Ablation: {ABLATION_FRAC*100:.0f}% subset, {ABLATION_EP} epochs each')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  DATASETS + TRANSFORMS  ██
# ══════════════════════════════════════════════════════════════════════════

class ICH9chDataset(Dataset):
    """Loads pre-stacked (380,380,9) float16 cache arrays from NB02."""
    def __init__(self, df, npy_root, transform, use_center_only=False):
        self.df = df.reset_index(drop=True)
        self.npy_root = Path(npy_root)
        self.transform = transform
        self.use_center_only = use_center_only

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = self.npy_root / f'{row["image_id"]}.npy'

        if p.exists():
            try:
                img = np.load(str(p)).astype(np.float32)  # expected (380,380,9)
            except Exception:
                img = np.zeros((BASE_IMG_SIZE, BASE_IMG_SIZE, 9), dtype=np.float32)
        else:
            img = np.zeros((BASE_IMG_SIZE, BASE_IMG_SIZE, 9), dtype=np.float32)

        if self.use_center_only:
            # center slice channels: [brain, subdural, bone]
            img = img[:, :, 3:6]

        if self.transform:
            img = self.transform(image=img)['image']

        labels = torch.tensor([row[c] for c in SUBTYPES], dtype=torch.float32)
        return img, labels


def get_transforms(target_size=380, augment=True):
    if augment:
        return A.Compose([
            A.RandomResizedCrop(size=(target_size, target_size), scale=(0.8, 1.0),
                                interpolation=1, p=1.0),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                               rotate_limit=15, p=0.5, border_mode=0),
            A.CoarseDropout(max_holes=8, max_height=48, max_width=48,
                            min_holes=1, min_height=8, min_width=8, fill=0, p=0.3),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(target_size, target_size),
        ToTensorV2(),
    ])

print('Datasets + transforms ready (pre-normalized cache, no runtime normalize).')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HELPERS: model + loss + hard mining + training loop  ██
# ══════════════════════════════════════════════════════════════════════════

def build_model(backbone=BACKBONE, in_ch=IN_CHANNELS, n_cls=N_CLASSES, pretrained=True):
    model = timm.create_model(backbone, pretrained=pretrained,
                              num_classes=0, drop_rate=0.4, drop_path_rate=0.2)
    old = model.conv_stem
    new = nn.Conv2d(in_ch, old.out_channels, kernel_size=old.kernel_size,
                    stride=old.stride, padding=old.padding,
                    bias=(old.bias is not None))
    k = in_ch // 3 if in_ch > 3 else 1
    with torch.no_grad():
        if in_ch == 3:
            new.weight.copy_(old.weight)
        else:
            new.weight.copy_(old.weight.repeat(1, k, 1, 1) / k)
        if old.bias is not None:
            new.bias.copy_(old.bias)
    model.conv_stem = new
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(model.num_features, n_cls)
    )
    return model


class MultiLabelLoss(nn.Module):
    def __init__(self, any_w=ANY_WEIGHT, sub_w=SUB_WEIGHT, ls=LABEL_SMOOTH):
        super().__init__()
        self.any_w = any_w
        self.sub_w = sub_w
        self.ls = ls

    def forward(self, logits, targets):
        t = targets * (1 - self.ls) + 0.5 * self.ls
        bce = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
        w = torch.ones_like(bce)
        w[:, 0] = self.any_w
        w[:, 1:] = self.sub_w
        return (bce * w).mean()


@torch.no_grad()
def mine_hard_negatives(model, dataset, device, top_k=MINE_TOP_K):
    model.eval()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                        num_workers=0, pin_memory=False)
    all_probs, all_labels = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):
            logits = model(imgs)
        all_probs.append(torch.sigmoid(logits[:, 0]).cpu())
        all_labels.append(labels[:, 0])

    probs = torch.cat(all_probs)
    labels = torch.cat(all_labels)
    neg_mask = labels < 0.5
    neg_probs = probs[neg_mask]
    neg_idxs = torch.arange(len(probs))[neg_mask]

    if len(neg_probs) == 0:
        return set()

    k = min(top_k, len(neg_probs))
    _, top_pos = neg_probs.topk(k)
    hard_set = set(neg_idxs[top_pos].tolist())
    return hard_set


def build_sampler(n_samples, hard_neg_indices=None):
    weights = np.ones(n_samples, dtype=np.float64)
    if hard_neg_indices:
        for idx in hard_neg_indices:
            if 0 <= idx < n_samples:
                weights[idx] *= MINE_BOOST
    return WeightedRandomSampler(weights, num_samples=n_samples, replacement=True)


def run_experiment(name, train_ds, val_ds, model, use_hard_mining=True,
                   lr=LR, n_ep=ABLATION_EP):
    print(f'\n{"─"*65}')
    print(f'Experiment: {name}')
    print(f'  Train={len(train_ds):,}, Val={len(val_ds):,}, hard_mining={use_hard_mining}')
    print(f'{"─"*65}')

    model = model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = MultiLabelLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    records = []
    best_auc = 0.0
    hard_neg_set = set()

    for ep in range(n_ep):
        if use_hard_mining and ep > 0:
            hard_neg_set = mine_hard_negatives(model, train_ds, DEVICE, top_k=MINE_TOP_K)

        sampler = build_sampler(len(train_ds), hard_neg_set if use_hard_mining else None)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=2, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                                num_workers=2, pin_memory=True)

        model.train()
        ep_loss, n_b = 0.0, 0
        for imgs, labels in tqdm(train_loader, desc=f'  Ep{ep}', leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                logits = model(imgs)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            ep_loss += loss.item()
            n_b += 1

        model.eval()
        all_p, all_l = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    all_p.append(torch.sigmoid(model(imgs)).cpu())
                all_l.append(labels)

        probs = torch.cat(all_p).numpy()
        labels_np = torch.cat(all_l).numpy()

        per_cls = [roc_auc_score(labels_np[:, i], probs[:, i])
                   for i in range(min(N_CLASSES, labels_np.shape[1]))
                   if labels_np[:, i].sum() > 0]
        auc = float(np.mean(per_cls)) if len(per_cls) else 0.5
        best_auc = max(best_auc, auc)

        print(f'  Ep{ep}: loss={ep_loss/max(n_b,1):.4f} auc_mean={auc:.4f} (best={best_auc:.4f})')
        records.append({'epoch': ep, 'loss': ep_loss/max(n_b,1), 'auc': auc})

        del train_loader, val_loader
        gc.collect()

    del model
    torch.cuda.empty_cache()
    gc.collect()
    return {'records': records, 'best_auc': best_auc}

print('Helpers ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  LOAD DATA + SUBSET  ██
# ══════════════════════════════════════════════════════════════════════════
manifest = pd.read_csv(MANIFEST_PATH)
train_full = manifest[manifest['fold'] != FOLD_ID].reset_index(drop=True)
val_full   = manifest[manifest['fold'] == FOLD_ID].reset_index(drop=True)

np.random.seed(SEED)
abl_idx = np.random.choice(len(train_full), size=int(len(train_full) * ABLATION_FRAC),
                           replace=False)
train_abl = train_full.iloc[abl_idx].reset_index(drop=True)

print(f'Ablation train: {len(train_abl):,} / {len(train_full):,}')
print(f'Val:            {len(val_full):,}')
print(f'Fold:           {FOLD_ID}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPERIMENT 1: Baseline (final config)  ██
# ══════════════════════════════════════════════════════════════════════════
results_all = {}
run_logs = {}

seed_everything(SEED)
baseline_train = ICH9chDataset(train_abl, NPY_CACHE_DIR,
                               get_transforms(target_size=380, augment=True),
                               use_center_only=False)
baseline_val = ICH9chDataset(val_full, NPY_CACHE_DIR,
                             get_transforms(target_size=380, augment=False),
                             use_center_only=False)

r_base = run_experiment(
    'Baseline: B4 + 9ch 2.5D + 380 + mining ON',
    baseline_train, baseline_val,
    build_model(backbone=BACKBONE, in_ch=9, n_cls=N_CLASSES),
    use_hard_mining=True
)
results_all['Baseline (final)'] = r_base['best_auc']
run_logs['Baseline (final)'] = r_base

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPERIMENT 2: Without 2.5D context (center slice only, 3ch)  ██
# ══════════════════════════════════════════════════════════════════════════
seed_everything(SEED)
no25d_train = ICH9chDataset(train_abl, NPY_CACHE_DIR,
                            get_transforms(target_size=380, augment=True),
                            use_center_only=True)
no25d_val = ICH9chDataset(val_full, NPY_CACHE_DIR,
                          get_transforms(target_size=380, augment=False),
                          use_center_only=True)

r_no25d = run_experiment(
    'No 2.5D: B4 + 3ch center-only + 380 + mining ON',
    no25d_train, no25d_val,
    build_model(backbone=BACKBONE, in_ch=3, n_cls=N_CLASSES),
    use_hard_mining=True
)
results_all['No 2.5D (3ch)'] = r_no25d['best_auc']
run_logs['No 2.5D (3ch)'] = r_no25d

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPERIMENT 3: 256 resolution instead of 380  ██
# ══════════════════════════════════════════════════════════════════════════
seed_everything(SEED)
res256_train = ICH9chDataset(train_abl, NPY_CACHE_DIR,
                             get_transforms(target_size=256, augment=True),
                             use_center_only=False)
res256_val = ICH9chDataset(val_full, NPY_CACHE_DIR,
                           get_transforms(target_size=256, augment=False),
                           use_center_only=False)

r_256 = run_experiment(
    '256 res: B4 + 9ch 2.5D + 256 + mining ON',
    res256_train, res256_val,
    build_model(backbone=BACKBONE, in_ch=9, n_cls=N_CLASSES),
    use_hard_mining=True
)
results_all['256 resolution'] = r_256['best_auc']
run_logs['256 resolution'] = r_256

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPERIMENT 4: Without hard negative mining  ██
# ══════════════════════════════════════════════════════════════════════════
seed_everything(SEED)
nomine_train = ICH9chDataset(train_abl, NPY_CACHE_DIR,
                             get_transforms(target_size=380, augment=True),
                             use_center_only=False)
nomine_val = ICH9chDataset(val_full, NPY_CACHE_DIR,
                           get_transforms(target_size=380, augment=False),
                           use_center_only=False)

r_no_mine = run_experiment(
    'No mining: B4 + 9ch 2.5D + 380 + mining OFF',
    nomine_train, nomine_val,
    build_model(backbone=BACKBONE, in_ch=9, n_cls=N_CLASSES),
    use_hard_mining=False
)
results_all['No mining'] = r_no_mine['best_auc']
run_logs['No mining'] = r_no_mine

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  ABLATION SUMMARY TABLE + CHART  ██
# ══════════════════════════════════════════════════════════════════════════

comparisons = [
    ('2.5D context',        'No 2.5D (3ch)',      results_all['No 2.5D (3ch)'],
     'Baseline (9ch 2.5D)', results_all['Baseline (final)']),
    ('Input resolution',    '256 resolution',     results_all['256 resolution'],
     'Baseline (380)',      results_all['Baseline (final)']),
    ('Hard-neg mining',     'No mining',          results_all['No mining'],
     'Mining ON (baseline)',results_all['Baseline (final)']),
]

print(f'\n{"Experiment":<18} {"Control":<22} {"AUC":>6}  {"Treatment":<22} {"AUC":>6}  {"Δ":>7}')
print('─' * 92)
for exp, ctrl_name, ctrl_auc, trt_name, trt_auc in comparisons:
    delta = trt_auc - ctrl_auc
    arrow = '↑' if delta > 0 else '↓' if delta < 0 else '→'
    print(f'{exp:<18} {ctrl_name:<22} {ctrl_auc:.4f}  {trt_name:<22} {trt_auc:.4f}  '
          f'{arrow}{abs(delta):.4f}')

print(f'\n⚠️  Constrained setting: {ABLATION_FRAC*100:.0f}% subset, {ABLATION_EP} epochs, fold {FOLD_ID}.')
print('   Use these as directional ablation signals, not final leaderboard claims.')

# Plot training curves
fig, ax = plt.subplots(figsize=(9, 4))
for name, result in run_logs.items():
    epochs = [r['epoch'] for r in result['records']]
    aucs   = [r['auc'] for r in result['records']]
    ax.plot(epochs, aucs, 'o-', label=f'{name} (best={result["best_auc"]:.4f})')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC (mean over classes)')
ax.set_title('Ablation Training Curves')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ablation_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
variants = list(results_all.keys())
aucs = [results_all[v] for v in variants]
colors = ['#1b5e20' if v == 'Baseline (final)' else '#66bb6a' for v in variants]
bars = ax.barh(variants, aucs, color=colors, edgecolor='white')
ax.set_title('Ablation AUC Results (baseline vs single-component removals)')
ax.set_xlim(0.5, 1.0)
for bar, val in zip(bars, aucs):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ablation_chart.png', dpi=150, bbox_inches='tight')
plt.show()

pd.DataFrame(comparisons,
             columns=['experiment', 'control', 'control_auc', 'treatment', 'treatment_auc']
).to_csv(OUTPUT_DIR / 'ablation_results.csv', index=False)

print('\nSaved: ablation_results.csv, ablation_chart.png, ablation_curves.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HEALTH CHECK  ██
# ══════════════════════════════════════════════════════════════════════════
errors = []
required_keys = ['Baseline (final)', 'No 2.5D (3ch)', '256 resolution', 'No mining']

for k in required_keys:
    if k not in results_all:
        errors.append(f'missing result: {k}')

if not (OUTPUT_DIR / 'ablation_results.csv').exists():
    errors.append('ablation_results.csv missing')
if not (OUTPUT_DIR / 'ablation_chart.png').exists():
    errors.append('ablation_chart.png missing')
if len(results_all) < 4:
    errors.append(f'Only {len(results_all)} results (expected 4)')

health = {
    'notebook': '02_ablation',
    'status': 'PASS' if not errors else 'FAIL',
    'errors': errors,
    'n_experiments': len(results_all),
    'results': {k: round(v, 5) for k, v in results_all.items()},
    'alignment': {
        'backbone': BACKBONE,
        'input_baseline': '9ch @ 380',
        'classes': N_CLASSES,
        'single_variable_ablation': True
    }
}
json.dump(health, open(OUTPUT_DIR / 'health_check_nb02.json', 'w'), indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:', errors)
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   {len(results_all)} experiments completed and aligned to final training recipe.')